In [8]:
# %pip install numpy
# %pip install pandas
# %pip install ultralytics
# %pip install pydevd-pycharm
# %pip install --upgrade debugpy pydevd

In [1]:
import cv2
import pandas as pd
import numpy as np
from ultralytics import YOLO
from collections import defaultdict
from scipy.spatial import distance
import matplotlib.pyplot as plt
import seaborn as sns
import copy

In [2]:
class VehicleTracker:
    def __init__(self, yolo_model_path, deepsort_config, video_path, output_path):
        self.yolo_model = YOLO(yolo_model_path)

        self.cap = cv2.VideoCapture(video_path)
        self.output_path = output_path
        self.fps = self.cap.get(cv2.CAP_PROP_FPS)
        self.frame_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.frame_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.out = cv2.VideoWriter(
            output_path,
            cv2.VideoWriter_fourcc(*"mp4v"),
            self.fps,
            (640, 368),
        )

        self.frame_count = 0
        self.vehicle_data = defaultdict(lambda: {"frames": 0, "track": [], "reids": 0, "last_seen": 0, "type" : "" , "frame_init": None, "frame_end": None, "time_init": None, "time_end": None })
        self.trajectories = {}
        self.vehicle_counts = defaultdict(int)
        self.density_per_frame = []
        self.heat_map = np.zeros((self.frame_height, self.frame_width), dtype=np.float32)
        self.last_frame_tracks = {}

    def process_frame_yolo(self, frame):
        results = self.yolo_model.track(frame, persist=True, verbose=False)
        current_frame_tracks = {}

        for result in results:
            if result is None or result.boxes is None or result.boxes.id is None:
                continue
            boxes = result.boxes.xyxy.cpu().numpy()
            track_ids = result.boxes.id.cpu().numpy()
            class_ids = result.boxes.cls.cpu().numpy()
            for box, track_id, class_id in zip(boxes, track_ids, class_ids):
                class_name = results[0].names[int(class_id)]

                reid = 0
                if track_id in self.last_frame_tracks and track_id != self.last_frame_tracks[track_id]:
                    reid = 1

                if track_id not in self.vehicle_data:
                    self.vehicle_data[track_id] = {"frames": 1, "track": [], "reids": 0, "last_seen": 0, "type" : class_name, "frame_init": self.frame_count, "frame_end": None, "time_init": self.frame_count/self.fps, "time_end": None}

                self.vehicle_data[track_id]["frames"] += 1
                if track_id not in self.vehicle_data:
                    self.vehicle_data[track_id] = {"frames": 1, "track": [], "reids": 0, "last_seen": 0, "type" : class_name}

                self.vehicle_data[track_id]["last_seen"] = self.frame_count
                self.vehicle_data[track_id]["type"] = class_name
                self.vehicle_data[track_id]["reids"] += reid
                self.vehicle_data[track_id]["frame_end"] = self.frame_count
                self.vehicle_data[track_id]["time_end"] = self.frame_count/self.fps

                x1, y1, x2, y2 = map(int, box)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, f"{track_id}:{class_name}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

                center_x = (x1 + x2) // 2
                center_y = (y1 + y2) // 2
                self.vehicle_data[track_id]["track"].append((center_x, center_y))
                self.trajectories[track_id] = self.trajectories.get(track_id, []) + [(center_x, center_y)] # Agrega la trayectoria a los registros
                current_frame_tracks[track_id] = track_id

        self.last_frame_tracks = current_frame_tracks # Actualizar los tracks de este frame

        # Estimación de densidad de tráfico
        self.density_per_frame.append(len(results))

        # Mapa de calor de trayectorias
        for track_id, data in self.trajectories.items():
            try:
                for x,y in data:
                    self.heat_map[y,x] += 1 # Incrementa en 1 la intensidad del pixel
            except:
                pass
        return frame

    def run(self):
        while True:
            ret, frame = self.cap.read()
            if not ret:
                break

            frame = cv2.resize(frame, (640, 368))

            self.frame_count += 1

            frame = self.process_frame_yolo(frame)

            counts = {}
            for track_id, data in self.vehicle_data.items():
                class_name = data["type"]
                if class_name in counts:
                    counts[class_name] += 1
                else:
                    counts[class_name] = 1

            # Dibujar conteo en el frame
            text_x, text_y = 10, 30  # Posición inicial del texto
            for class_name, count in counts.items():
                text = f"{class_name}: {count}"
                cv2.putText(frame, text, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                text_y += 30

            # self.out.write(frame)
            # cv2.imshow("frame", frame)
            plt.imshow(frame)
            plt.axis("off")
            plt.pause(0.01)

        self.cap.release()
        self.out.release()
        cv2.destroyAllWindows()

    def get_vehicle_data(self):
        return self.vehicle_data

    def get_density_per_frame(self):
        return self.density_per_frame

    def get_heat_map(self):
        return self.heat_map

    def get_trajectories(self):
        return self.trajectories

    def get_frame_count(self):
        return self.frame_count

In [2]:
# Configuración del modelo YOLO
tracker1 = VehicleTracker(
    "../models/yolov8s.pt",
    {},
    "../media/vehicles.mp4",
    "../media/output_yolo1.mp4"

)
tracker1.run()

Unable to load extension pydevd_plugins.extensions.types.pydevd_plugin_pandas_types
Could not connect to 127.0.0.1: 43269
Traceback (most recent call last):
  File "/home/vsalcedop/.pycharm_helpers/pydev/_pydevd_bundle/pydevd_comm.py", line 467, in start_client
    s.connect((host, port))
ConnectionRefusedError: [Errno 111] Connection refused
Traceback (most recent call last):
  File "/home/vsalcedop/.pycharm_helpers/jupyter_debug/pydev_jupyter_utils.py", line 81, in attach_to_debugger
    debugger.connect(pydev_localhost.get_localhost(), debugger_port)
  File "/home/vsalcedop/.pycharm_helpers/pydev/pydevd.py", line 704, in connect
    s = start_client(host, port)
        ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vsalcedop/.pycharm_helpers/pydev/_pydevd_bundle/pydevd_comm.py", line 467, in start_client
    s.connect((host, port))
ConnectionRefusedError: [Errno 111] Connection refused
Failed to connect to target debugger.
